In [29]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
df=pd.read_csv(r"C:\Users\KIIT0001\Desktop\Python\CSV\diabetes.csv")
X=df.drop('Outcome',axis='columns')
columns_with_zero=X.columns[(X == 0).any()].tolist()
try:
    columns_with_zero.remove('Pregnancies')
except Exception:
    print("No column with such name exist")

df[columns_with_zero]=df[columns_with_zero].replace(0,np.nan)
impute=SimpleImputer(missing_values=np.nan,strategy="mean")
X=impute.fit_transform(X)

Y=df.Outcome.apply(lambda x:"Diabetic" if x==1 else "Non-Diabetic")

X_train, X_test, y_train, y_test =train_test_split(X,Y,test_size=0.25,shuffle=True,stratify=Y,random_state=42)
def objective(trial):
    n_estimators=trial.suggest_int('n_estimators',50,200)
    max_depth=trial.suggest_int('max_depth',3,20)
    model=RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    score=cross_val_score(model,X=X_train,y=y_train,scoring='accuracy').mean()
    return score


study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=50)

print(study.best_params)
print(study.best_value)
best_model=RandomForestClassifier(**study.best_params,random_state=42)
best_model.fit(X_train,y_train)
y_pred=best_model.predict(X_test)
print(f"Accuracy of the best model:{accuracy_score(y_test,y_pred)*100:.2f}%")

[I 2026-07-16 12:47:30,624] A new study created in memory with name: no-name-bf737896-d4b4-43dc-bae2-37670b8820e3


[I 2026-07-16 12:47:31,351] Trial 0 finished with value: 0.7691004497751124 and parameters: {'n_estimators': 76, 'max_depth': 19}. Best is trial 0 with value: 0.7691004497751124.
[I 2026-07-16 12:47:31,853] Trial 1 finished with value: 0.7621439280359821 and parameters: {'n_estimators': 52, 'max_depth': 18}. Best is trial 0 with value: 0.7691004497751124.
[I 2026-07-16 12:47:33,008] Trial 2 finished with value: 0.7673613193403298 and parameters: {'n_estimators': 128, 'max_depth': 20}. Best is trial 0 with value: 0.7691004497751124.
[I 2026-07-16 12:47:34,701] Trial 3 finished with value: 0.7656371814092953 and parameters: {'n_estimators': 148, 'max_depth': 18}. Best is trial 0 with value: 0.7691004497751124.
[I 2026-07-16 12:47:37,430] Trial 4 finished with value: 0.7569415292353823 and parameters: {'n_estimators': 104, 'max_depth': 3}. Best is trial 0 with value: 0.7691004497751124.
[I 2026-07-16 12:47:41,935] Trial 5 finished with value: 0.7656371814092953 and parameters: {'n_estimat

{'n_estimators': 87, 'max_depth': 16}
0.777751124437781
Accuracy of the best model:77.08%
